In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, avg, count, sum as spark_sum

spark = SparkSession.builder \
    .appName("Tugas4")\
    .master("local[*]")\
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("SprakSession berhasil dibuat")

26/09/16 22:19:06 WARN Utils: Your hostname, nabill-IdeaPad-3-14IML05 resolves to a loopback address: 127.0.1.1; using 192.168.11.111 instead (on interface wlp0s20f3)
26/09/16 22:19:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 22:19:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SprakSession berhasil dibuat


In [2]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/nabill/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/nabill/tugas4/transaksi_september_2026.csv


In [3]:
#1. Membaca dan Eksplorasi Awal
df_tugas = spark.read.csv (
    "hdfs://localhost:9000/user/nabill/tugas4/transaksi_september_2026.csv",
    header=True, inferSchema=True
)

df_tugas.printSchema()
print("Jumlah Baris: ", df_tugas.count())
df_tugas.show(10)


root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah Baris:  1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semarang|       

In [4]:
#2. Menangani Data Kosong
jumlah_null = df_tugas.filter(col("rating").isNull()).count()
print(f" Jumlah Baris dengan rating kosong (NULL): {jumlah_null}")

rata_rating = df_tugas.select(avg("rating")).first()[0]
df_clean = df_tugas.na.fill({"rating" : rata_rating})

print(f"Nilai pengganti (rata-rata rating): {rata_rating:.2f}")
print("Jumlah rating NULL setelah ditangani: ", df_clean.filter(col("rating").isNull()).count())

 Jumlah Baris dengan rating kosong (NULL): 204
Nilai pengganti (rata-rata rating): 4.15
Jumlah rating NULL setelah ditangani:  0


Alasan : Metode df.na.fill() dengan pengisian nilai rata-rata lebih dipilih dari pada df.na.drop karena kalau df.na.drop dipakai maka akan kehilangan kira kira 200 baris data yang bisa mengurangi keakuratan analisis variabel seperti transaksi, kota, dan pendapatan. Jadi kalau pakai df.na.fill() struktur distribusi data tetap terjaga tanpa membuang informasi transaksi penting

In [5]:
#3. Transformasi Data
df_transformed = df_clean \
    .withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan")) \
    .withColumn("tier_transaksi", when(col("total_pendapatan") > 50000, "Besar").otherwise("Kecil"))

df_transformed.select("order_id", "unit_terjual", "harga_satuan", "total_pendapatan", "tier_transaksi").show(10)

+--------+------------+------------+----------------+--------------+
|order_id|unit_terjual|harga_satuan|total_pendapatan|tier_transaksi|
+--------+------------+------------+----------------+--------------+
|ORD-3000|           3|       90000|          270000|         Besar|
|ORD-3001|           3|      200000|          600000|         Besar|
|ORD-3002|           8|       60000|          480000|         Besar|
|ORD-3003|           6|      350000|         2100000|         Besar|
|ORD-3004|          10|       60000|          600000|         Besar|
|ORD-3005|           5|       20000|          100000|         Besar|
|ORD-3006|           2|       20000|           40000|         Kecil|
|ORD-3007|           8|       90000|          720000|         Besar|
|ORD-3008|           7|       20000|          140000|         Besar|
|ORD-3009|          10|       90000|          900000|         Besar|
+--------+------------+------------+----------------+--------------+
only showing top 10 rows



In [6]:
#4. Analisis dengan GroupBy
kategori_pendapatan = df_transformed.groupBy("kategori") \
    .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
    .orderBy(col("total_pendapatan").desc())
kategori_pendapatan.show(1)

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row



In [7]:
kota_tier_besar = df_transformed.filter(col("tier_transaksi") == "Besar") \
    .groupBy("kota") \
    .agg(count("order_id").alias("jumlah_transaksi_besar")) \
    .orderBy(col("jumlah_transaksi_besar").desc())

kota_tier_besar.show(1)

+----+----------------------+
|kota|jumlah_transaksi_besar|
+----+----------------------+
|Solo|                   179|
+----+----------------------+
only showing top 1 row



In [8]:
rating_per_metode = df_transformed.groupBy("metode_pembayaran") \
    .agg(avg("rating").alias("rata_rata_rating")) \
    .orderBy(col("rata_rata_rating").desc())

rating_per_metode.show()

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD| 4.167310656870009|
|    Transfer Bank|   4.1592349097265|
|         E-Wallet| 4.137728643216084|
|     Kartu Kredit|4.1179474608816475|
+-----------------+------------------+



In [9]:
#5. Menyimpan hasil ke HDFS
output_hdfs_path = "hdfs://localhost:9000/user/nabill/tugas4/hasil_transaksi_september"
df_transformed.write.csv(output_hdfs_path, header=True, mode="overwrite")
print("Hasil berhasil disimpan ke HDFS")

Hasil berhasil disimpan ke HDFS


In [ ]:
df_verifikasi = spark.read.csv(output_hdfs_path, header=True)
print("Jumlah baris verifikasi daro HDFS: " df_verifikasi.count()